# Gemma 4 pure-VLM pipeline (Gemma 4 detects and explains)

In [ ]:
# 1. Check GPU - verify CUDA is available and print GPU name and VRAM
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')


In [ ]:
# 2. Install packages - install all required Python libraries
!pip install -q torch accelerate
!pip install -q "transformers>=5.10.1" pillow
!pip install -q json-repair


In [ ]:
# 4. Imports - load all Python libraries used across this notebook
import os, json, re, time, tempfile
from pathlib import Path
from collections import Counter
from PIL import Image
import torch
import transformers
try:
    from json_repair import repair_json
except ImportError:
    pass  # available after cell 2 install; used only in the repair cell


In [ ]:
# Load HuggingFace token - read API token from Kaggle secrets
import os, transformers
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret('HUGGINGFACE_TOKEN')
os.environ['HF_TOKEN'] = hf_token
transformers.logging.set_verbosity_error()
print('Token loaded.')

In [ ]:
# ── Config + run mode + path resolution ──────────────────────────────
import glob
from pathlib import Path
from collections import Counter

IMAGES_DIR = '/kaggle/input/datasets/deividasbuinevicius/visdrone-dataset/VisDrone2019-DET-val/images'

MODEL_ID = 'google/gemma-4-E2B-it'

# ── Mode switch ──────────────────────────────────────────────────────
# 'detect'         → boxes only          (RQ1 detection metrics)
# 'detect+explain' → boxes + explanation (RQ2 qualitative)
RUN_MODE = 'detect+explain'

MAX_IMAGES         = 0    # 0 = all 548 val images
MAX_DETS_PER_IMAGE = 15   # explain only the top-15 objects; ALL objects are still detected
OUTPUT_DIR         = '/kaggle/working'

if not (Path(IMAGES_DIR).is_dir() and list(Path(IMAGES_DIR).glob('*.jpg'))):
    hits = glob.glob('/kaggle/input/**/VisDrone2019-DET-val/images', recursive=True)
    if hits:
        IMAGES_DIR = hits[0]
    else:
        dirs = Counter(str(Path(p).parent) for p in glob.glob('/kaggle/input/**/*.jpg', recursive=True))
        if dirs:
            IMAGES_DIR = dirs.most_common(1)[0][0]
    print('Discovered val folder:', IMAGES_DIR)

n_imgs = len(list(Path(IMAGES_DIR).glob('*.jpg')))
print('Images dir exists:', Path(IMAGES_DIR).is_dir())
print('Images found     :', n_imgs, '(expected 548 for VisDrone2019-DET-val)')
print('Run mode         :', RUN_MODE)
assert n_imgs > 0, 'No images found - attach the VisDrone dataset or fix IMAGES_DIR.'

In [ ]:
# 7. Load model - download and initialise Gemma 4 pipeline on GPU
from transformers import pipeline, GenerationConfig

vqa_pipe = pipeline(
    task='image-text-to-text',
    model=MODEL_ID,
    device_map='auto',
    torch_dtype=torch.float16,
    token=hf_token,
)

gen_config = GenerationConfig.from_pretrained(MODEL_ID)
# detect-only output is short (label + 4 numbers per box); detect+explain needs more room
gen_config.max_new_tokens = 2048 if RUN_MODE == 'detect' else 3072
gen_kwargs = dict(generation_config=gen_config)

print('Gemma 4 loaded. Memory used:', round(torch.cuda.memory_allocated() / 1e9, 2), 'GB')
print(f'max_new_tokens: {gen_config.max_new_tokens}  (mode: {RUN_MODE})')


In [ ]:
# 8. Define helper functions - prompts, inference runner, and JSON parser
import json, re, tempfile
from pathlib import Path
from PIL import Image

VISDRONE_CLASSES = [
    'pedestrian', 'people', 'bicycle', 'car', 'van',
    'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor'
]
CLASS_TO_ID = {c: i for i, c in enumerate(VISDRONE_CLASSES)}


def _run_pipe(pil_image, prompt_text):
    tmp = tempfile.NamedTemporaryFile(suffix='.jpg', delete=False, dir='/tmp')
    tmp.close()
    pil_image.convert('RGB').save(tmp.name, format='JPEG', quality=95)
    messages = [
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': [
            {'type': 'image', 'url': tmp.name},
            {'type': 'text',  'text': prompt_text},
        ]}
    ]
    output = vqa_pipe(messages, return_full_text=False, generate_kwargs=gen_kwargs)
    os.unlink(tmp.name)
    torch.cuda.empty_cache()
    return re.sub(r'<\s*/?\s*(?:end_of_)?turn[^>]*>', '', output[0]['generated_text']).strip()


# ── Prompt: detection only ────────────────────────────────────────────────────
def build_detection_prompt():
    classes = ', '.join(VISDRONE_CLASSES)
    example = (
        '[\n'
        '  {"label": "car", "box_2d": [200, 300, 250, 400]},\n'
        '  {"label": "pedestrian", "box_2d": [100, 150, 180, 200]}\n'
        ']'
    )
    return (
        'This is an aerial drone image taken from above.\n\n'
        f'Detect all visible objects from these categories: {classes}.\n\n'
        'Return a JSON array where EACH detected object is a SEPARATE entry with:\n'
        '  - "label": one of the category names above\n'
        '  - "box_2d": [y1, x1, y2, x2] normalised to 0-1000 (top-left = 0,0)\n\n'
        'Example format:\n'
        + example + '\n\n'
        'Do NOT group multiple objects under one entry. One object per entry. '
        'Return ONLY a valid JSON array. If no objects are visible, return [].'
    )


# ── Prompt: detection + explanation ──────────────────────────────────────────
def build_combined_prompt():
    classes = ', '.join(VISDRONE_CLASSES)
    example = (
        '[\n'
        '  {"label": "car", "box_2d": [200, 300, 250, 400], "explanation": "A white car is parked at the roadside, its rectangular shape clearly visible from above."},\n'
        '  {"label": "pedestrian", "box_2d": [100, 150, 180, 200], "explanation": "A person in dark clothing is walking along the pavement."}\n'
        ']'
    )
    return (
        'This is an aerial drone image taken from above.\n\n'
        f'Detect all visible objects from these categories: {classes}.\n\n'
        'Return a JSON array. EACH object MUST be a separate entry containing '
        'ALL THREE fields below, in this order:\n'
        '  - "label": one of the category names above\n'
        '  - "box_2d": [y1, x1, y2, x2] normalised to 0-1000 (top-left = 0,0)\n'
        '  - "explanation": a single sentence describing its colour, appearance, and what it is doing.\n\n'
        'The "explanation" field is REQUIRED for every object and must never be empty or omitted. '
        'Do not return a detection-only array.\n\n'
        'Example format:\n'
        + example + '\n\n'
        'Do NOT group multiple objects under one entry. One object per entry. '
        'Return ONLY a valid JSON array. If no objects are visible, return [].'
    )


def get_prompt():
    return build_detection_prompt() if RUN_MODE == 'detect' else build_combined_prompt()


# ── Parser: handles both modes (explanation field optional) ───────────────────
def parse_response(response, img_w, img_h, debug=False):
    if debug:
        print('[DEBUG] raw response:')
        print(response)
        print('[DEBUG] ---')

    match = re.search(r'```json\s*(.*?)\s*```', response, re.DOTALL)
    raw = match.group(1) if match else None
    if not raw:
        match = re.search(r'\[.*\]', response, re.DOTALL)
        raw = match.group() if match else None
    if not raw:
        if debug:
            print('[DEBUG] no JSON array found')
        return [], 'no_json_found'
    try:
        detections = json.loads(raw)
    except json.JSONDecodeError as e:
        if debug:
            print('[DEBUG] JSONDecodeError:', e, '-- attempting repair')
        try:
            from json_repair import repair_json
            detections = json.loads(repair_json(raw))
            if debug:
                print('[DEBUG] repair succeeded')
        except Exception as e2:
            if debug:
                print('[DEBUG] repair failed:', e2)
            return [], 'json_parse_error'

    results = []
    for det in detections:
        label = str(det.get('label', '')).lower().strip()
        if label not in CLASS_TO_ID:
            if debug:
                print('[DEBUG] unknown label skipped:', repr(label))
            continue
        box_2d = det.get('box_2d', [])
        if len(box_2d) != 4:
            continue
        y1_n, x1_n, y2_n, x2_n = [int(v) for v in box_2d]
        x1 = max(0, min(x1_n / 1000 * img_w, img_w))
        y1 = max(0, min(y1_n / 1000 * img_h, img_h))
        x2 = max(0, min(x2_n / 1000 * img_w, img_w))
        y2 = max(0, min(y2_n / 1000 * img_h, img_h))
        if x2 <= x1 or y2 <= y1:
            continue
        results.append({
            'class_id':    CLASS_TO_ID[label],
            'class_name':  label,
            'bbox':        [x1, y1, x2, y2],
            'explanation': str(det.get('explanation', '')).strip(),
        })

    # MAX_DETS_PER_IMAGE caps how many objects are EXPLAINED, not how many are
    # detected: keep every box (for RQ1 detection metrics) and only blank the
    # explanation text beyond the cap.
    if MAX_DETS_PER_IMAGE > 0:
        for _d in results[MAX_DETS_PER_IMAGE:]:
            _d['explanation'] = ''
    return results, 'ok'


print('Helper functions defined.')
print('Active prompt:', 'detection only' if RUN_MODE == 'detect' else 'detection + explanation')


In [ ]:
# 9. Smoke test - run on 2 images to verify the pipeline before the full run
# Smoke test on 2 images
image_paths = sorted(Path(IMAGES_DIR).glob('*.jpg'))[:2]

for image_path in image_paths:
    print('\n===', image_path.name, '===')
    img = Image.open(image_path).convert('RGB')

    response = _run_pipe(img, get_prompt())
    print('Raw response (first 400 chars):', response[:400])

    detections, status = parse_response(response, img.width, img.height, debug=True)
    print(f'Status: {status} | Detections: {len(detections)}')

    for d in detections:
        bbox_str = str([int(v) for v in d['bbox']])
        if RUN_MODE == 'detect+explain':
            print(f"  {d['class_name']:20s} bbox={bbox_str}  ->  {d['explanation']}")
        else:
            print(f"  {d['class_name']:20s} bbox={bbox_str}")

print('\nSmoke test complete.')


In [ ]:
# 10. Full run - process all 548 val images, log timing, and checkpoint every 10
DEBUG = False
import time

all_image_paths = sorted(Path(IMAGES_DIR).glob('*.jpg'))
if MAX_IMAGES and MAX_IMAGES > 0:
    all_image_paths = all_image_paths[:MAX_IMAGES]

print(f'Processing {len(all_image_paths)} images in mode: {RUN_MODE}')

prompt        = get_prompt()

# Resume from checkpoint: reload partials so a re-run continues instead of
# restarting. detect+explain on 548 can exceed a 12h Kaggle session, so finish
# in a second run. REQUIRES Session options > Persistence = "Files only",
# otherwise /kaggle/working (and the checkpoint) is wiped when the session ends.
results_all   = {}
timing_log    = {}
_partial  = f'{OUTPUT_DIR}/gemma4_purevlm_partial.json'
_tpartial = f'{OUTPUT_DIR}/gemma4_timing_partial.json'
if Path(_partial).is_file():
    results_all = json.load(open(_partial))
    if Path(_tpartial).is_file():
        timing_log = json.load(open(_tpartial))
    print(f'Resuming: {len(results_all)} images already done -> will skip them.')
predictions   = {
    name: [{'class_id': d['class_id'], 'class_name': d['class_name'],
            'confidence': d.get('confidence', 1.0), 'bbox': d['bbox']} for d in dets]
    for name, dets in results_all.items()
}
parse_errors  = 0

run_start = time.perf_counter()

for i, image_path in enumerate(all_image_paths):
    if image_path.name in results_all:
        continue   # already done in a previous session
    print(f'[{i+1}/{len(all_image_paths)}] {image_path.name}', end=' ', flush=True)

    img = Image.open(image_path).convert('RGB')

    t0 = time.perf_counter()
    response = _run_pipe(img, prompt)
    detections, status = parse_response(response, img.width, img.height, debug=DEBUG)
    inference_time = time.perf_counter() - t0

    if status != 'ok':
        parse_errors += 1
        timing_log[image_path.name] = {'inference_time_s': round(inference_time, 3), 'num_detections': 0}
        print(f'PARSE ERROR ({status}) [{inference_time:.1f}s]')
        results_all[image_path.name] = []
        predictions[image_path.name] = []
        continue

    timing_log[image_path.name] = {
        'inference_time_s': round(inference_time, 3),
        'num_detections':   len(detections),
    }

    enriched = [
        {
            'class_id':    d['class_id'],
            'class_name':  d['class_name'],
            'confidence':  1.0,
            'bbox':        d['bbox'],
            'explanation': d['explanation'],
        }
        for d in detections
    ]

    print(f'{len(enriched)} detections [{inference_time:.1f}s]')
    results_all[image_path.name] = enriched
    predictions[image_path.name] = [
        {'class_id': d['class_id'], 'class_name': d['class_name'],
         'confidence': d['confidence'], 'bbox': d['bbox']}
        for d in enriched
    ]

    if (i + 1) % 10 == 0:
        elapsed = time.perf_counter() - run_start
        avg     = elapsed / (i + 1)
        eta     = avg * (len(all_image_paths) - i - 1)
        print(f'  -- {i+1} done | avg {avg:.1f}s/img | ETA {eta/60:.0f} min --')
        with open(f'{OUTPUT_DIR}/gemma4_purevlm_partial.json', 'w') as f:
            json.dump(results_all, f, indent=2)
        with open(f'{OUTPUT_DIR}/gemma4_timing_partial.json', 'w') as f:
            json.dump(timing_log, f, indent=2)

total_elapsed = time.perf_counter() - run_start
inf_times = [v['inference_time_s'] for v in timing_log.values()]
avg_inf   = sum(inf_times) / len(inf_times) if inf_times else 0

print(f'\nDone. Parse errors: {parse_errors}/{len(all_image_paths)}')
print(f'Total wall time:     {total_elapsed/60:.1f} min')
print(f'Avg inference time:  {avg_inf:.2f}s/img')


In [ ]:
# 11. Save results - write predictions, explanations, and timing JSON to Drive
explanations_path = f'{OUTPUT_DIR}/gemma4_purevlm_explanations.json'
with open(explanations_path, 'w') as f:
    json.dump({
        'model':   MODEL_ID,
        'setting': 'pure-vlm',
        'mode':    RUN_MODE,
        'config':  {'max_images': MAX_IMAGES, 'max_dets_per_image': MAX_DETS_PER_IMAGE},
        'images':  results_all,
    }, f, indent=2)

predictions_path = f'{OUTPUT_DIR}/gemma4_purevlm_predictions.json'
with open(predictions_path, 'w') as f:
    json.dump(predictions, f, indent=2)

timing_path = f'{OUTPUT_DIR}/gemma4_timing.json'
with open(timing_path, 'w') as f:
    json.dump({
        'model':   MODEL_ID,
        'mode':    RUN_MODE,
        'note':    'Single inference call per image (detection only or detection+explanation depending on RUN_MODE).',
        'summary': {
            'num_images':           len(timing_log),
            'total_elapsed_min':    round(total_elapsed / 60, 2),
            'avg_inference_time_s': round(avg_inf, 3),
        },
        'per_image': timing_log,
    }, f, indent=2)

total_detections = sum(len(v) for v in results_all.values())
print(f'Mode: {RUN_MODE}')
print(f'Saved {len(results_all)} images, {total_detections} total detections')
print(f'Explanations: {explanations_path}')
print(f'Predictions:  {predictions_path}')
print(f'Timing:       {timing_path}')


## 11-12. Check results & repair parse errors
Run after the main loop. Cell 11 reports parse errors; cell 12 re-runs only those images.

In [ ]:
# 12. Check results - report detection statistics and list images with 0 detections
import json
from pathlib import Path
from collections import Counter

with open(f'{OUTPUT_DIR}/gemma4_purevlm_predictions.json') as f:
    _preds = json.load(f)
with open(f'{OUTPUT_DIR}/gemma4_timing.json') as f:
    _timing = json.load(f)

det_counts  = {name: len(dets) for name, dets in _preds.items()}
empty       = [name for name, n in det_counts.items() if n == 0]
total_dets  = sum(det_counts.values())
avg_inf     = _timing['summary'].get('avg_inference_time_s', '?')

print(f'Images in file    : {len(_preds)} / 548')
print(f'Total detections  : {total_dets}')
print(f'Avg per image     : {total_dets / max(len(_preds),1):.1f}')
print(f'Avg inference     : {avg_inf}s')
print(f'Images with 0 det : {len(empty)}  ({len(empty)/max(len(_preds),1)*100:.1f}%)')

if empty:
    print('\nImages to repair:')
    for name in sorted(empty)[:20]:
        print(f'  {name}')
    if len(empty) > 20:
        print(f'  ... and {len(empty)-20} more')
else:
    print('\n✓ No parse errors - ready to run evaluate.py.')

dist = Counter(det_counts.values())
print('\nDetection count distribution:')
for n in sorted(dist)[:12]:
    print(f'  {n:3d} dets: {dist[n]} images')


In [ ]:
# 13. Repair - re-run images with parse errors and patch the saved files
# Only runs if there are images to fix; safe to run even if check showed none.
import time
from json_repair import repair_json

with open(f'{OUTPUT_DIR}/gemma4_purevlm_predictions.json')  as f: _preds = json.load(f)
with open(f'{OUTPUT_DIR}/gemma4_purevlm_explanations.json') as f: _expls = json.load(f)
with open(f'{OUTPUT_DIR}/gemma4_timing.json')               as f: _timing = json.load(f)

_results_all = _expls.get('images', {})
_timing_log  = _timing.get('per_image', {})

failed = [
    p for p in sorted(Path(IMAGES_DIR).glob('*.jpg'))
    if _preds.get(p.name) == [] or p.name not in _preds
]

if not failed:
    print('Nothing to repair.')
else:
    print(f'Re-running {len(failed)} images...')
    _prompt   = get_prompt()
    repaired  = 0
    still_bad = 0

    for i, image_path in enumerate(failed):
        print(f'[{i+1}/{len(failed)}] {image_path.name}', end=' ', flush=True)
        img = Image.open(image_path).convert('RGB')

        t0 = time.perf_counter()
        raw_resp = _run_pipe(img, _prompt)
        elapsed  = time.perf_counter() - t0

        # Try normal parse first, then repair
        match = re.search(r'```json\s*(.*?)\s*```', raw_resp, re.DOTALL) or re.search(r'\[.*\]', raw_resp, re.DOTALL)
        raw_json = match.group(1) if (match and match.lastindex) else (match.group() if match else None)
        dets = []
        if raw_json:
            try:
                dets = json.loads(raw_json)
            except json.JSONDecodeError:
                try:
                    dets = json.loads(repair_json(raw_json))
                    print('[repaired]', end=' ')
                except Exception:
                    pass

        parsed = []
        for det in dets:
            label = str(det.get('label', '')).lower().strip()
            if label not in CLASS_TO_ID: continue
            box_2d = det.get('box_2d', [])
            if len(box_2d) != 4: continue
            y1_n, x1_n, y2_n, x2_n = [int(v) for v in box_2d]
            x1 = max(0, min(x1_n/1000*img.width,  img.width))
            y1 = max(0, min(y1_n/1000*img.height, img.height))
            x2 = max(0, min(x2_n/1000*img.width,  img.width))
            y2 = max(0, min(y2_n/1000*img.height, img.height))
            if x2 <= x1 or y2 <= y1: continue
            parsed.append({'class_id': CLASS_TO_ID[label], 'class_name': label,
                           'bbox': [x1,y1,x2,y2], 'explanation': str(det.get('explanation','')).strip()})

        if not parsed:
            still_bad += 1
            print(f'still failed [{elapsed:.1f}s]')
            continue

        repaired += 1
        if MAX_DETS_PER_IMAGE > 0:
            for _d in parsed[MAX_DETS_PER_IMAGE:]:
                _d['explanation'] = ''
        enriched = [{'class_id': d['class_id'], 'class_name': d['class_name'],
                     'confidence': 1.0, 'bbox': d['bbox'], 'explanation': d['explanation']}
                    for d in parsed]
        _results_all[image_path.name] = enriched
        _preds[image_path.name]       = [{'class_id': d['class_id'], 'class_name': d['class_name'],
                                           'confidence': d['confidence'], 'bbox': d['bbox']} for d in enriched]
        _timing_log[image_path.name]  = {'inference_time_s': round(elapsed,3),
                                          'num_detections': len(parsed), 'repaired': True}
        print(f'{len(parsed)} detections [{elapsed:.1f}s]')

    # Save patched files
    _expls['images']      = _results_all
    _timing['per_image']  = _timing_log
    with open(f'{OUTPUT_DIR}/gemma4_purevlm_predictions.json',  'w') as f: json.dump(_preds,  f, indent=2)
    with open(f'{OUTPUT_DIR}/gemma4_purevlm_explanations.json', 'w') as f: json.dump(_expls,  f, indent=2)
    with open(f'{OUTPUT_DIR}/gemma4_timing.json',               'w') as f: json.dump(_timing, f, indent=2)

    print(f'\nRepaired: {repaired}  |  Still failed: {still_bad}')
    print('Files updated in place.')
